# Interactive demonstration of the precision-recall tradeoff

This demonstration illustrates how the classification threshold affects
precision and recall, and how the precision-recall (PR) curve depends
on the discriminative ability of a classifier.

## Dataset

The dataset consists of 120 two-dimensional observations drawn from
two overlapping Gaussian distributions:

$$
\mathbf{x}\mid y=0 \sim
\mathcal{N}\left(
\begin{bmatrix}0\\0\end{bmatrix},\,0.25\mathbf{I}
\right)
$$

$$
\mathbf{x}\mid y=1 \sim
\mathcal{N}\left(
\begin{bmatrix}1\\1\end{bmatrix},\,0.25\mathbf{I}
\right)
$$

## Classifiers

We consider two logistic regression classifiers of the form

$$
p(y=1\mid\mathbf{x}) =
\sigma(\mathbf{w}^{T}\mathbf{x}+b),
$$

where $\sigma(z)=1/(1+e^{-z})$ is the sigmoid function.

Classifier A has a decision boundary with a slope of $-1$,
which is optimal for the underlying Gaussian distributions.
Classifier B has a suboptimal slope of $-0.2$.

An observation is classified as positive if

$$
p(y=1\mid\mathbf{x})\geq\tau,
$$

where $\tau\in[0,1]$ is the classification threshold.

For $0<\tau<1$, the decision boundary is given by

$$
x_2=-\frac{w_1}{w_2}x_1+
\frac{\log\left(\frac{\tau}{1-\tau}\right)-b}{w_2}.
$$

Changing the threshold therefore translates the decision boundary
without changing its orientation.

## Precision and recall

Precision and recall are defined as

$$
\mathrm{Precision}=\frac{TP}{TP+FP},
\qquad
\mathrm{Recall}=\frac{TP}{TP+FN}.
$$

- **Precision:** The proportion of predicted positive observations
  that are actually positive.
- **Recall:** The proportion of actual positive observations that
  are correctly classified as positive.

An **operating point** is the pair of precision and recall values
obtained at a particular classification threshold $\tau$.

The precision-recall (PR) curve shows how precision and recall
vary as the classification threshold is changed from 0 to 1.

The colored markers on the PR curves indicate the current
operating point of each classifier.

## Explore the visualization

The blue points represent the negative class ($y=0$), and the
orange points represent the positive class ($y=1$).

For each classifier, observations above its decision boundary
are classified as positive, while observations below it are
classified as negative.

Use the slider to move the decision boundaries and consider
the following questions:

- Which points (color and position relative to the decision
  boundary) correspond to true positives, false positives,
  true negatives, and false negatives?

- Move the decision boundary downwards by decreasing the
  threshold. What happens to the number of orange and blue
  points classified as positive? How does this affect precision
  and recall?

- Move the decision boundary upwards by increasing the
  threshold. Can you achieve perfect precision? What happens
  to recall when you do so?

- Can you find a threshold at which recall is 100%? Why does
  this not necessarily imply that precision is also 100%?

- Can you find an example where increasing the threshold
  decreases precision? Explain what happens in terms of
  the individual orange and blue points crossing the boundary.

- Compare the two classifiers at a threshold of 0.5.
  How does the orientation of the decision boundary affect
  the number of false positives and false negatives?

- Use the PR curves to compare the two classifiers at the
  same recall rather than at the same threshold. Which
  classifier achieves higher precision, and why?

**Key takeaway:** Changing the classification threshold
moves the operating point along a PR curve, whereas changing
the classifier can change the PR curve itself.

In [ ]:
#@title Interactive precision-recall demonstration { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from scipy.special import expit, logit
from sklearn.metrics import precision_recall_curve
from matplotlib.patches import Circle
from IPython.display import display
from google.colab import output

output.enable_custom_widget_manager()

# --------------------------------------------------
# 1. Fixed dataset
# --------------------------------------------------

# Reproduce the exact data points from the chat demo
i = np.arange(120)
y = (i >= 60).astype(int)

u = ((i * 73 + 19) % 127 + 0.5) / 127
v = ((i * 47 + 31) % 131 + 0.5) / 131
r = 0.5 * np.sqrt(-2 * np.log(u))

X = np.column_stack([
    y + r * np.cos(2 * np.pi * v),
    y + r * np.sin(2 * np.pi * v)
])

# --------------------------------------------------
# 2. Two fixed logistic regression classifiers
# --------------------------------------------------

classifiers = [
    {"name": "Classifier A", "slope": -1.0,
     "color": "#168c83", "linestyle": "-"},

    {"name": "Classifier B", "slope": -0.2,
     "color": "#a855c7", "linestyle": "--"}
]

for clf in classifiers:

    # Normal vector perpendicular to decision boundary
    w = np.array([-clf["slope"], 1.0])

    # Same weight magnitude for both classifiers
    w = 4 * np.sqrt(2) * w / np.linalg.norm(w)

    # Boundary passes through (0.5, 0.5) at threshold 0.5
    b = -w @ np.array([0.5, 0.5])

    clf["w"] = w
    clf["b"] = b

    # Logistic regression probabilities
    clf["scores"] = expit(X @ w + b)

    # Full precision-recall curve
    clf["precision"], clf["recall"], _ = (
        precision_recall_curve(y, clf["scores"])
    )

# --------------------------------------------------
# 3. Visualization
# --------------------------------------------------

from matplotlib.colors import LinearSegmentedColormap

BLUE = "#347bd3"
ORANGE = "#e07836"

def plot(threshold=0.5):

    fig = plt.figure(figsize=(13, 10))

    gs = fig.add_gridspec(
        2, 2,
        height_ratios=[1.5, 1],
        hspace=0.35,
        wspace=0.25
    )

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    cm_axes = [
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 1])
    ]

    # ==============================================
    # TOP LEFT: Data and decision boundaries
    # ==============================================

    for k, color in [(0, BLUE), (1, ORANGE)]:

        # Gaussian contours
        for radius, alpha in [(0.5, 0.45), (1.0, 0.25)]:
            ax1.add_patch(Circle(
                (k, k), radius,
                fill=False,
                edgecolor=color,
                linestyle="--",
                linewidth=1.2,
                alpha=alpha
            ))

        # Individual observations
        ax1.scatter(
            X[y == k, 0],
            X[y == k, 1],
            c=color,
            s=32,
            alpha=0.85,
            edgecolors="none",
            label=f"Class {k}",
            zorder=3
        )

    x1 = np.linspace(-2, 3, 200)

    for clf, ax_cm in zip(classifiers, cm_axes):

        w = clf["w"]
        b = clf["b"]
        color = clf["color"]
        style = clf["linestyle"]

        # Decision boundary
        if 0 < threshold < 1:

            x2 = (logit(threshold) - b - w[0]*x1) / w[1]

            ax1.plot(
                x1, x2,
                color=color,
                linestyle=style,
                linewidth=2.5,
                label=f'{clf["name"]} '
                      f'(slope {clf["slope"]})',
                zorder=4
            )

        # Binary predictions
        pred = clf["scores"] >= threshold

        TP = np.sum((pred == 1) & (y == 1))
        FP = np.sum((pred == 1) & (y == 0))
        TN = np.sum((pred == 0) & (y == 0))
        FN = np.sum((pred == 0) & (y == 1))

        precision = TP / (TP + FP) if TP + FP else np.nan
        recall = TP / (TP + FN)

        # ==========================================
        # TOP RIGHT: Precision-recall curves
        # ==========================================

        ax2.plot(
            clf["recall"],
            clf["precision"],
            color=color,
            linestyle=style,
            linewidth=2.5,
            label=clf["name"]
        )

        # Current operating point
        if not np.isnan(precision):

            ax2.scatter(
                recall, precision,
                color=color,
                s=110,
                edgecolors="white",
                linewidth=1.5,
                zorder=5
            )

        # ==========================================
        # BOTTOM: Confusion matrix
        # ==========================================

        cm = np.array([
            [TN, FP],
            [FN, TP]
        ])

        labels = np.array([
            ["TN", "FP"],
            ["FN", "TP"]
        ])

        # Classifier-specific colormap
        cmap = LinearSegmentedColormap.from_list(
            clf["name"], ["#ffffff", color]
        )

        # Fixed color scale across classifiers and thresholds
        ax_cm.imshow(
            cm,
            cmap=cmap,
            vmin=0,
            vmax=max(np.bincount(y)),
            aspect="equal"
        )

        # Cell annotations
        for row in range(2):
            for col in range(2):

                count = cm[row, col]
                text_color = (
                    "white" if count > 35 else "#222222"
                )

                ax_cm.text(
                    col, row - 0.09,
                    str(count),
                    ha="center",
                    va="center",
                    fontsize=24,
                    fontweight="bold",
                    color=text_color
                )

                ax_cm.text(
                    col, row + 0.19,
                    labels[row, col],
                    ha="center",
                    va="center",
                    fontsize=11,
                    color=text_color
                )

        # Matrix formatting
        ax_cm.set_xticks([0, 1], labels=["0", "1"])
        ax_cm.set_yticks([0, 1], labels=["0", "1"])

        ax_cm.xaxis.tick_top()
        ax_cm.xaxis.set_label_position("top")

        ax_cm.set_xlabel("Predicted class", labelpad=8)
        ax_cm.set_ylabel("True class")

        ax_cm.set_title(
            clf["name"],
            color=color,
            fontsize=14,
            fontweight="bold",
            pad=15
        )

        # White separators between cells
        ax_cm.axhline(0.5, color="white", linewidth=2)
        ax_cm.axvline(0.5, color="white", linewidth=2)

        ax_cm.tick_params(length=0)

        for spine in ax_cm.spines.values():
            spine.set_visible(False)

        # Precision and recall below each matrix
        precision_text = (
            f"{precision:.1%}"
            if not np.isnan(precision)
            else "Undefined"
        )

        ax_cm.text(
            0.5, -0.08,
            f"Precision (TP / (TP + FP)): {precision_text}\n"
            f"Recall (TP / (TP + FN)): {recall:.1%}",
            transform=ax_cm.transAxes,
            ha="center",
            va="top",
            fontsize=13,
            fontweight="medium",
            color=color,
            linespacing=1.5
        )

    # ==============================================
    # Plot formatting
    # ==============================================

    ax1.set(
        xlim=(-2, 3),
        ylim=(-2, 3),
        xlabel=r"$x_1$",
        ylabel=r"$x_2$",
        title="Decision boundaries"
    )

    ax1.set_aspect("equal")
    ax1.set_xticks(np.arange(-2, 4))
    ax1.set_yticks(np.arange(-2, 4))
    ax1.legend(loc="upper left", fontsize=9)

    ax2.set(
        xlim=(0, 1.02),
        ylim=(0, 1.02),
        xlabel="Recall",
        ylabel="Precision",
        title="Precision-recall curves"
    )

    ax2.set_aspect("equal")
    ax2.set_xticks(np.linspace(0, 1, 6))
    ax2.set_yticks(np.linspace(0, 1, 6))
    ax2.legend(loc="lower left", fontsize=10)

    for ax in [ax1, ax2]:
        ax.grid(alpha=0.15)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        f"Classification threshold: {threshold:.2f}",
        fontsize=17
    )

    fig.subplots_adjust(
        top=0.92,
        bottom=0.12
    )

    plt.show()
    plt.close(fig)

# --------------------------------------------------
# 4. Interactive threshold slider
# --------------------------------------------------

slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description="Threshold:",
    continuous_update=True,
    readout_format=".2f",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "initial"}
)

widgets.interact(plot, threshold=slider);

interactive(children=(FloatSlider(value=0.5, description='Threshold:', layout=Layout(width='90%'), max=1.0, st…